# Premier League Match Score Prediction
### By: Nadim Sherif

This project explores the use of machine learning to predict Premier League match scores using historical data. Match data from the 2019/2020 to 2023/2024 seasons is used to train predictive models, which are then evaluated on the 2024/2025 season. Model performance is measured both by exact score accuracy and by correctly predicting match results, providing insight into the strengths and limitations of statistical approaches to football match prediction.


In [81]:
import pandas as pd

data = pd.read_csv('epl_final.csv')
data = data.drop(columns=['HalfTimeHomeGoals', 'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeCorners','AwayCorners', 'HomeFouls', 'AwayFouls', 'HomeRedCards', 'AwayRedCards'])


Since the data is in "match" format, we will need to convert it into "team" format. This will make each row specific to a teams performance in a certain game.

In [82]:
home = data[['MatchDate', 'Season', 'HomeTeam', 'FullTimeHomeGoals', 'FullTimeAwayGoals']] \
    .rename(columns={
        'HomeTeam': 'Team',
        'FullTimeHomeGoals': 'GoalsFor',
        'FullTimeAwayGoals': 'GoalsAgainst'
    })

home['IsHome'] = 1

away = data[['MatchDate', 'Season', 'AwayTeam', 'FullTimeAwayGoals', 'FullTimeHomeGoals']] \
    .rename(columns={
        'AwayTeam': 'Team',
        'FullTimeAwayGoals': 'GoalsFor',
        'FullTimeHomeGoals': 'GoalsAgainst'
    })

away['IsHome'] = 0

team_games = pd.concat([home, away]).sort_values('MatchDate')


Want to check if the team is in form, so we will find the rolling average of the previous 5 games.

In [83]:
team_games['AvgGoalsFor_5'] = (
    team_games.groupby('Team')['GoalsFor']
    .shift(1)
    .rolling(5)
    .mean()
)

team_games['AvgGoalsAgainst_5'] = (
    team_games.groupby('Team')['GoalsAgainst']
    .shift(1)
    .rolling(5)
    .mean()
)

print(team_games.tail())

       MatchDate   Season            Team  GoalsFor  GoalsAgainst  IsHome  \
9375  2025-05-04  2024/25       Brentford         4             3       1   
9378  2025-05-04  2024/25       Liverpool         1             3       0   
9377  2025-05-04  2024/25       Tottenham         1             1       0   
9379  2025-05-05  2024/25  Crystal Palace         1             1       1   
9379  2025-05-05  2024/25   Nott'm Forest         1             1       0   

      AvgGoalsFor_5  AvgGoalsAgainst_5  
9375            2.2                1.0  
9378            3.0                1.2  
9377            2.6                2.2  
9379            2.6                2.0  
9379            2.0                2.0  


Now, this data is merged back into the initial data.

In [84]:
home_features = team_games[team_games['IsHome'] == 1][
    ['MatchDate', 'Team', 'AvgGoalsFor_5', 'AvgGoalsAgainst_5']
].rename(columns={
    'Team': 'HomeTeam',
    'AvgGoalsFor_5': 'HomeAvgGF_5',
    'AvgGoalsAgainst_5': 'HomeAvgGA_5'
})

away_features = team_games[team_games['IsHome'] == 0][
    ['MatchDate', 'Team', 'AvgGoalsFor_5', 'AvgGoalsAgainst_5']
].rename(columns={
    'Team': 'AwayTeam',
    'AvgGoalsFor_5': 'AwayAvgGF_5',
    'AvgGoalsAgainst_5': 'AwayAvgGA_5'
})

data = data.merge(home_features, on=['MatchDate', 'HomeTeam'], how='left')
data = data.merge(away_features, on=['MatchDate', 'AwayTeam'], how='left')

print(data.tail())

       Season   MatchDate        HomeTeam       AwayTeam  FullTimeHomeGoals  \
9375  2024/25  2025-05-04       Brentford     Man United                  4   
9376  2024/25  2025-05-04        Brighton      Newcastle                  1   
9377  2024/25  2025-05-04        West Ham      Tottenham                  1   
9378  2024/25  2025-05-04         Chelsea      Liverpool                  3   
9379  2024/25  2025-05-05  Crystal Palace  Nott'm Forest                  1   

      FullTimeAwayGoals FullTimeResult  HomeShots  AwayShots  \
9375                  3              H         12         14   
9376                  1              D          5         13   
9377                  1              D         11          7   
9378                  1              H         17         11   
9379                  1              D         20         12   

      HomeShotsOnTarget  AwayShotsOnTarget  HomeYellowCards  AwayYellowCards  \
9375                  6                  5                0 

In [85]:
train_seasons = ['2019/20', '2020/21', '2021/22', '2022/23', '2023/24']
test_seasons = ['2024/25']

train_data = data[data['Season'].isin(train_seasons)]
test_data = data[data['Season'].isin(test_seasons)]

print(f"Training data shape: {train_data.shape}")
print(f"Testing data shape: {test_data.shape}")
print(train_data.head())

Training data shape: (1900, 17)
Testing data shape: (350, 17)
       Season   MatchDate        HomeTeam          AwayTeam  \
7130  2019/20  2019-08-09       Liverpool           Norwich   
7131  2019/20  2019-08-10        West Ham          Man City   
7132  2019/20  2019-08-10     Bournemouth  Sheffield United   
7133  2019/20  2019-08-10         Burnley       Southampton   
7134  2019/20  2019-08-10  Crystal Palace           Everton   

      FullTimeHomeGoals  FullTimeAwayGoals FullTimeResult  HomeShots  \
7130                  4                  1              H         15   
7131                  0                  5              A          5   
7132                  1                  1              D         13   
7133                  3                  0              H         10   
7134                  0                  0              D          6   

      AwayShots  HomeShotsOnTarget  AwayShotsOnTarget  HomeYellowCards  \
7130         12                  7                  

In [86]:
data.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeYellowCards',
       'AwayYellowCards', 'HomeAvgGF_5', 'HomeAvgGA_5', 'AwayAvgGF_5',
       'AwayAvgGA_5'],
      dtype='object')

### Defining Features / Targets

In [87]:
y_train_goals = train_data[['FullTimeHomeGoals', 'FullTimeAwayGoals']]
y_test_goals  = test_data[['FullTimeHomeGoals', 'FullTimeAwayGoals']]

y_train_result = train_data['FullTimeResult']
y_test_result  = test_data['FullTimeResult']

In [88]:
features = [
    'HomeAvgGF_5', 'HomeAvgGA_5',
    'AwayAvgGF_5', 'AwayAvgGA_5',
    'HomeYellowCards', 'AwayYellowCards'
]

X_train = train_data[features]
X_test  = test_data[features]

X_test.head()

,HomeAvgGF_5,HomeAvgGA_5,AwayAvgGF_5,AwayAvgGA_5,HomeYellowCards,AwayYellowCards
9030,2.2,1.4,2.8,1.4,2,3
9031,1.8,2.4,1.8,2.0,3,1
9032,2.6,1.0,1.4,2.2,2,2
9033,1.6,2.6,1.0,3.0,1,1
9034,2.6,1.8,1.4,2.6,2,4


In [89]:
from sklearn.ensemble import RandomForestRegressor

goal_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

goal_model.fit(X_train, y_train_goals)

goal_preds = goal_model.predict(X_test)


In [90]:
from sklearn.ensemble import RandomForestClassifier

result_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

result_model.fit(X_train, y_train_result)

result_preds = result_model.predict(X_test)


In [91]:
import numpy as np

exact_score_correct = np.sum(
    (goal_preds[:, 0].round() == y_test_goals['FullTimeHomeGoals'].values) &
    (goal_preds[:, 1].round() == y_test_goals['FullTimeAwayGoals'].values)
)

exact_score_accuracy = exact_score_correct / len(y_test_goals)

print(f"Exact score accuracy: {exact_score_accuracy:.3f}")


Exact score accuracy: 0.077


In [92]:
from sklearn.metrics import accuracy_score

result_accuracy = accuracy_score(y_test_result, result_preds)

print(f"Result accuracy: {result_accuracy:.3f}")


Result accuracy: 0.383


In [93]:
from sklearn.metrics import mean_absolute_error

mae_home = mean_absolute_error(
    y_test_goals['FullTimeHomeGoals'],
    goal_preds[:, 0]
)

mae_away = mean_absolute_error(
    y_test_goals['FullTimeAwayGoals'],
    goal_preds[:, 1]
)

print(f"Home goals MAE: {mae_home:.2f}")
print(f"Away goals MAE: {mae_away:.2f}")


Home goals MAE: 1.11
Away goals MAE: 0.95


In [94]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test_result, result_preds))
print(classification_report(y_test_result, result_preds))


[[43 10 65]
 [24 13 51]
 [48 18 78]]
              precision    recall  f1-score   support

           A       0.37      0.36      0.37       118
           D       0.32      0.15      0.20        88
           H       0.40      0.54      0.46       144

    accuracy                           0.38       350
   macro avg       0.36      0.35      0.34       350
weighted avg       0.37      0.38      0.37       350

